In [100]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
from pathlib import Path

# time series decomposition
from statsmodels.tsa.seasonal import seasonal_decompose

# anomaly detection models
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from scipy import stats

# llm for the report agent
import anthropic

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [101]:
DATA_DIR = Path('project_anomaly_detection')

# load both datasets
viaggiatori = pd.read_csv(DATA_DIR / 'TIPOLOGIA_VIAGGIATORE.csv')
allarmi = pd.read_csv(DATA_DIR / 'ALLARMI.csv')

print('viaggiatori:', viaggiatori.shape)
print('allarmi:', allarmi.shape)

viaggiatori: (5095, 33)
allarmi: (5080, 24)


In [102]:
# quick look at the traveler dataset
viaggiatori.head()

,NAZIONALITA,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,GIORNO_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,CITTA_PARTENZA,CODICE_PAESE_ARR,CODICE_PAESE_PART,PAESE_ARR,PAESE_PART,ZONA,ENTRATI,INVESTIGATI,ALLARMATI,TIPO_DOCUMENTO,GENERE,FASCIA_ETA,FLAG_TRANSITO,COMPAGNIA_AEREA,NUMERO_VOLO,ESITO_CONTROLLO,note_operatore,codice_rischio,Tipo Documento,FASCIA ETA,3nazionalita,compagnia%aerea,num volo
0,ALB,NAP,DUR,2024,02,13,2024-02-13 07:30:00,Napoli Capodichino,King Shaka International,Napoli,Durban,ITA,ZAF,Italia,Sudafrica,6,1,1,0,Passaporto,F,N.D.,Singola Tratta,Fly Dubai,FZ1681,RESPINTO,NaN,NaN,Passaporto,N.D.,ALB,Fly Dubai,FZ1681
1,NaN,FCO,JFK,2024,01,22,2024-01-22 16:35:00,Fiumicino,John F Kennedy International,Roma,New York,ITA,USA,Italia,Stati Uniti,5,1,0,1,Carta d'identità,F,18-30,Singola Tratta,ITA Airways,AZ0609,NaN,NaN,NaN,Carta d'identità,18-30,ALB,ITA Airways,AZ0609
2,ALB,TSF,TIA,2024,02,4,2024-02-04 20:10:00,Treviso-Sant'Angelo,Rinas Mother Teresa,Treviso,Tirana,ITA,ALB,Italia,Albania,4,58,58,13,N.D.,f,31-45,Singola Tratta,Ryanair DAC,FR8400,SEGNALATO,NaN,NaN,N.D.,31-45,ALB,Ryanair DAC,FR8400
3,AFG,FCO,IST,2024,01,25,2024-01-25 13:05:00,Fiumicino,Havalimani,Roma,Istanbul,IT,TR,Italia,Turchia,5,1,1,0,N.D.,M,61+,Singola Tratta,Turkish Airlines,TK1865,NaN,NaN,NaN,N.D.,61+,AFG,Turkish Airlines,TK1865
4,ALB,BGY,MLE,2024,02,13,FEB 13 2024,Orio al Serio,Male International,Bergamo,Male,ITA,MDV,Italia,Maldive,2,2,2,1,Permesso di soggiorno,F,46-60,Singola Tratta,Fly Dubai,FZ1571,SEGNALATO,NaN,NaN,Permesso di soggiorno,46-60,ALB,Fly Dubai,FZ1571


In [103]:
viaggiatori.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5095 entries, 0 to 5094
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   NAZIONALITA            4979 non-null   object
 1   AREOPORTO_ARRIVO       5095 non-null   object
 2   AREOPORTO_PARTENZA     5095 non-null   object
 3   ANNO_PARTENZA          5095 non-null   object
 4   MESE_PARTENZA          5095 non-null   object
 5   GIORNO_PARTENZA        5095 non-null   int64 
 6   DATA_PARTENZA          5095 non-null   object
 7   DESCR_AEREOPORTO_ARR   5095 non-null   object
 8   DESCR_AEREOPORTO_PART  5095 non-null   object
 9   CITTA_ARR              5095 non-null   object
 10  CITTA_PARTENZA         5095 non-null   object
 11  CODICE_PAESE_ARR       5095 non-null   object
 12  CODICE_PAESE_PART      5095 non-null   object
 13  PAESE_ARR              5095 non-null   object
 14  PAESE_PART             5095 non-null   object
 15  ZONA                 

In [104]:
# quick look at the alarms dataset
allarmi.head()

,OCCORRENZE,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,CITTA_PARTENZA,CODICE_PAESE_ARR,CODICE_PAESE_PART,PAESE_ARR,PAESE_PART,ZONA,TOT,MOTIVO_ALLARME,note_operatore,flag_rischio,Paese Partenza,CODICE PAESE ARR,3zona,paese%arr,tot voli
0,Voli con Allarmi,FCO,IST,2024,01,2024-01-30 09:15:00,Fiumicino,Havalimani,Roma,Istanbul,ITA,TUR,Italia,Turchia,5,1,Manuale,NaN,NaN,Turchia,ITA,5,Italia,1
1,Viaggiatori con Allarmi,CIA,STN,2024,02,2024-02-03 13:15:00,Ciampino,Stansted,Roma,Londra,ITA,GBR,Italia,Regno Unito,5,5,Manuale,NaN,NaN,Regno Unito,ITA,5,Italia,5
2,Viaggiatori entrati nel Sistema,FCO,LHR,2024,01,2024-01-15 08:45:00,Fiumicino,London Heathrow,Roma,Londra,ITA,GBR,Italia,Regno Unito,5,110,TSC,NaN,NaN,Regno Unito,ITA,5,Italia,110
3,Voli con Allarmi,MXP,LHR,2024,02,2024-02-02 08:40:00,Malpensa,London Heathrow,Milano,Londra,IT,GB,Italia,Regno Unito,2,1,SDI,NaN,NaN,Regno Unito,ITA,2,Italia,1
4,Viaggiatori con Allarmi,PSA,BRS,2024,02,2024-02-16 12:50:00,Galileo Galilei,Bristol,Pisa,Bristol,ITA,GBR,Italia,Regno Unito,8,2,INTERPOL,NaN,NaN,Regno Unito,ITA,8,Italia,2


In [105]:
allarmi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5080 entries, 0 to 5079
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   OCCORRENZE             5080 non-null   object
 1   AREOPORTO_ARRIVO       5080 non-null   object
 2   AREOPORTO_PARTENZA     5080 non-null   object
 3   ANNO_PARTENZA          5080 non-null   object
 4   MESE_PARTENZA          5080 non-null   object
 5   DATA_PARTENZA          5080 non-null   object
 6   DESCR_AEREOPORTO_ARR   5080 non-null   object
 7   DESCR_AEREOPORTO_PART  4971 non-null   object
 8   CITTA_ARR              5080 non-null   object
 9   CITTA_PARTENZA         4979 non-null   object
 10  CODICE_PAESE_ARR       5080 non-null   object
 11  CODICE_PAESE_PART      5026 non-null   object
 12  PAESE_ARR              5080 non-null   object
 13  PAESE_PART             5006 non-null   object
 14  ZONA                   5080 non-null   object
 15  TOT                  

## Cleaning - TIPOLOGIA_VIAGGIATORE

In [106]:
df = viaggiatori.copy()

# all the ways null is encoded in this dataset, found in the dataset all the possible variant to NULL
NULL_VALUES = ['N.D.', 'n.d.', 'ND', 'N/D', 'N/A', 'n/a', '?', '??', '???',
               '//', '-', 'null', 'NULL', 'unknown', 'Unknown', 'UNKN', 'UNK',
               ' ', '', 'ZZ', 'XX', 'EU']

df.replace(NULL_VALUES, np.nan, inplace=True)

# the last 5 columns are helper/cleaned versions added externally, drop them
df.drop(columns=['Tipo Documento', 'FASCIA ETA', '3nazionalita', 'compagnia%aerea', 'num volo'], inplace=True)

print(df.shape)

(5095, 28)


In [107]:
# fix ANNO: only the bad encodings get corrected to 2024, MESE stays untouched
anno_corrections = {'24': '2024', 'anno 2024': '2024', '2023': '2024', '2024.': '2024'}
df['ANNO_PARTENZA'] = df['ANNO_PARTENZA'].replace(anno_corrections).astype(int)

# fix MESE: Italian text months to numeric
italian_months = {'GEN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAG': 5, 'GIU': 6,
                  'LUG': 7, 'AGO': 8, 'SET': 9, 'OTT': 10, 'NOV': 11, 'DIC': 12}
df['MESE_PARTENZA'] = df['MESE_PARTENZA'].replace(italian_months).astype(float).astype('Int64')

# parse DATA_PARTENZA: dataset has many different formats mixed together
italian_months_long = {'GEN': 'Jan', 'FEB': 'Feb', 'MAR': 'Mar', 'APR': 'Apr',
                       'MAG': 'May', 'GIU': 'Jun', 'LUG': 'Jul', 'AGO': 'Aug',
                       'SET': 'Sep', 'OTT': 'Oct', 'NOV': 'Nov', 'DIC': 'Dec'}

def parse_date(val):
    if pd.isna(val):
        return pd.NaT
    val = str(val).strip()
    for ita, eng in italian_months_long.items():
        val = val.replace(ita, eng)
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S', '%Y-%m-%d',
                '%Y/%m/%d', '%d.%m.%Y', '%d-%m-%y', '%b %d %Y'):
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            pass
    try:
        return pd.to_datetime(val, dayfirst=True)
    except:
        return pd.NaT

df['DATA_PARTENZA'] = df['DATA_PARTENZA'].apply(parse_date)

print('ANNO unique:', df['ANNO_PARTENZA'].unique())
print('MESE unique:', sorted(df['MESE_PARTENZA'].dropna().unique().tolist()))
print(df['DATA_PARTENZA'].isna().sum(), 'unparsed dates')

ANNO unique: [2024]
MESE unique: [1, 2, 12]
0 unparsed dates


In [108]:
# clean ENTRATI, INVESTIGATI, ALLARMATI: strip text artifacts, keep the number
def extract_number(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    val = val.replace(',', '.').replace('~', '').replace('pax', '').replace('voli', '').strip()
    try:
        return float(val)
    except:
        return np.nan

df['ENTRATI']     = df['ENTRATI'].apply(extract_number)
df['INVESTIGATI'] = df['INVESTIGATI'].apply(extract_number)
df['ALLARMATI']   = df['ALLARMATI'].apply(extract_number)

# negative values are impossible, treat as null
df.loc[df['INVESTIGATI'] < 0, 'INVESTIGATI'] = np.nan
df.loc[df['ALLARMATI'] < 0, 'ALLARMATI'] = np.nan

print(df[['ENTRATI', 'INVESTIGATI', 'ALLARMATI']].describe())

            ENTRATI  INVESTIGATI    ALLARMATI
count   5010.000000  5008.000000  5017.000000
mean      42.266267    41.564497     7.466813
std      248.559256   227.369899    71.786689
min     -500.000000     0.000000     0.000000
25%        1.000000     1.000000     0.000000
50%        3.000000     2.000000     1.000000
75%       76.000000    75.000000    10.000000
max    10000.000000  9999.000000  5000.000000


In [109]:
# clean ENTRATI, INVESTIGATI, ALLARMATI: strip text artifacts, keep the number
def extract_number(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    val = val.replace(',', '.').replace('~', '').replace('pax', '').strip()
    try:
        return float(val)
    except:
        return np.nan

df['ENTRATI']    = df['ENTRATI'].apply(extract_number)
df['INVESTIGATI'] = df['INVESTIGATI'].apply(extract_number)
df['ALLARMATI']  = df['ALLARMATI'].apply(extract_number)

# negative or zero investigators is impossible, treat as null
df.loc[df['INVESTIGATI'] < 0, 'INVESTIGATI'] = np.nan
df.loc[df['ALLARMATI'] < 0, 'ALLARMATI'] = np.nan

print(df[['ENTRATI', 'INVESTIGATI', 'ALLARMATI']].describe())

            ENTRATI  INVESTIGATI    ALLARMATI
count   5010.000000  5008.000000  5017.000000
mean      42.266267    41.564497     7.466813
std      248.559256   227.369899    71.786689
min     -500.000000     0.000000     0.000000
25%        1.000000     1.000000     0.000000
50%        3.000000     2.000000     1.000000
75%       76.000000    75.000000    10.000000
max    10000.000000  9999.000000  5000.000000


### Standardizing categorical columns

Four cleaning operations in this cell:

**1. GENERE → M / F / NaN**  
The column has many variants for the same value (`f`, `Femmina`, `Female` → `F`; `m`, `Maschio`, `Male` → `M`). Anything that doesn't map to either is set to NaN (e.g. `X`, `N/B`, `1`, `2`).

**2. Airport codes → uppercase**  
IATA codes like `Bgy`, `vce`, `Pmo` are standardized to `BGY`, `VCE`, `PMO`. This is needed later when grouping by route.

**3. Country codes: 2-letter → 3-letter (ISO 3166-1 alpha-3)**  
Some rows use the 2-letter standard (`IT`, `AL`, `GB`) and others use 3-letter (`ITA`, `ALB`, `GBR`). We unify everything to alpha-3 to avoid treating the same country as two different values.

| Alpha-2 | Alpha-3 |
|---------|---------|
| IT | ITA |
| AL | ALB |
| TR | TUR |
| AE | ARE |
| GB | GBR |

**4. NAZIONALITA → valid 3-letter code or NaN**  
After uppercasing, any value that is not exactly 3 characters long is invalid (e.g. `//`, `EU`, `UNKN`) and is set to NaN.

In [110]:
# normalize GENERE to M/F/NaN
female_vals = {'f', 'femmina', 'female', 'donna', 'f.'}
male_vals   = {'m', 'maschio', 'male', 'uomo', 'male'}

def normalize_gender(val):
    if pd.isna(val):
        return np.nan
    v = str(val).strip().lower()
    if v in female_vals:
        return 'F'
    if v in male_vals:
        return 'M'
    return np.nan

df['GENERE'] = df['GENERE'].apply(normalize_gender)

# uppercase airport codes and fix mixed case
df['AREOPORTO_ARRIVO'] = df['AREOPORTO_ARRIVO'].str.strip().str.upper()
df['AREOPORTO_PARTENZA'] = df['AREOPORTO_PARTENZA'].str.strip().str.upper()

# fix 2-letter ISO country codes to 3-letter
iso2_to_iso3 = {'IT': 'ITA', 'AL': 'ALB', 'TR': 'TUR', 'AE': 'ARE',
                'GB': 'GBR', 'EG': 'EGY', 'DE': 'DEU', 'FR': 'FRA'}

df['CODICE_PAESE_ARR']  = df['CODICE_PAESE_ARR'].str.strip().str.upper().replace(iso2_to_iso3)
df['CODICE_PAESE_PART'] = df['CODICE_PAESE_PART'].str.strip().str.upper().replace(iso2_to_iso3)

# clean NAZIONALITA: strip and uppercase, anything not a valid 3-letter code becomes NaN
df['NAZIONALITA'] = df['NAZIONALITA'].str.strip().str.upper()
df.loc[df['NAZIONALITA'].str.len() != 3, 'NAZIONALITA'] = np.nan

print('GENERE:', df['GENERE'].value_counts(dropna=False).to_dict())
print('CODICE_PAESE_ARR unique:', df['CODICE_PAESE_ARR'].unique())
print('NAZIONALITA nulls:', df['NAZIONALITA'].isna().sum())

GENERE: {'F': 2226, 'M': 2195, nan: 674}
CODICE_PAESE_ARR unique: ['ITA']
NAZIONALITA nulls: 321


In [111]:
# standardize FASCIA_ETA: fix text labels and invalid values
fascia_map = {
    'minore': '0-17',
    'adulto': '31-45',
    '101+':   '61+',
    '-5':     np.nan,
    'N/C':    np.nan,
}
df['FASCIA_ETA'] = df['FASCIA_ETA'].replace(fascia_map)

print(df['FASCIA_ETA'].value_counts(dropna=False).to_string())

FASCIA_ETA
46-60    943
31-45    922
18-30    916
0-17     904
61+      884
NaN      526


In [112]:
# final check: null counts per column
null_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
print(null_pct[null_pct > 0].round(1).to_string())
print('cleaned shape:', df.shape)

viaggiatori_clean = df.copy()

codice_rischio     99.2
note_operatore     98.8
TIPO_DOCUMENTO     35.5
ESITO_CONTROLLO    25.3
GENERE             13.2
FASCIA_ETA         10.3
NAZIONALITA         6.3
COMPAGNIA_AEREA     5.1
NUMERO_VOLO         4.0
INVESTIGATI         1.7
ENTRATI             1.7
ALLARMATI           1.5
ZONA                0.0
cleaned shape: (5095, 28)


## Cleaning - ALLARMI

In [113]:
df2 = allarmi.copy()

df2.replace(NULL_VALUES, np.nan, inplace=True)

# drop helper columns added externally
df2.drop(columns=['Paese Partenza', 'CODICE PAESE ARR', '3zona', 'paese%arr', 'tot voli'], inplace=True)

# OCCORRENZE: '???', 'N/C', 'ALLARMATI' are not valid categories
df2['OCCORRENZE'] = df2['OCCORRENZE'].replace({'???': np.nan, 'N/C': np.nan, 'ALLARMATI': np.nan})

print(df2.shape)

(5080, 19)


In [114]:
# same fixes as viaggiatori: ANNO, MESE, DATA_PARTENZA
df2['ANNO_PARTENZA'] = df2['ANNO_PARTENZA'].replace(anno_corrections).astype(int)

df2['MESE_PARTENZA'] = df2['MESE_PARTENZA'].replace(italian_months).astype(float).astype('Int64')

df2['DATA_PARTENZA'] = df2['DATA_PARTENZA'].apply(parse_date)

print('ANNO unique:', df2['ANNO_PARTENZA'].unique())
print('MESE unique:', sorted(df2['MESE_PARTENZA'].dropna().unique().tolist()))
print(df2['DATA_PARTENZA'].isna().sum(), 'unparsed dates')

ANNO unique: [2024]
MESE unique: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
0 unparsed dates


In [115]:
# TOT: same logic as the numeric columns in viaggiatori
# has '-500', '-99', '-1' (impossible negatives) and '0 voli', '1 voli' (text artifacts)
df2['TOT'] = df2['TOT'].apply(extract_number)
df2.loc[df2['TOT'] < 0, 'TOT'] = np.nan
df2.loc[df2['TOT'] != df2['TOT'].round(), 'TOT'] = np.nan  # 0.5 is not a valid count

print(df2['TOT'].describe())

count     4960.000000
mean        85.174194
std       2018.266690
min          0.000000
25%          1.000000
50%          2.000000
75%         29.000000
max      99999.000000
Name: TOT, dtype: float64


In [116]:
# uppercase airport codes and fix country codes (same as viaggiatori)
df2['AREOPORTO_ARRIVO']   = df2['AREOPORTO_ARRIVO'].str.strip().str.upper()
df2['AREOPORTO_PARTENZA'] = df2['AREOPORTO_PARTENZA'].str.strip().str.upper()

df2['CODICE_PAESE_ARR']  = df2['CODICE_PAESE_ARR'].str.strip().str.upper().replace(iso2_to_iso3)
df2['CODICE_PAESE_PART'] = df2['CODICE_PAESE_PART'].str.strip().str.upper().replace(iso2_to_iso3)

# final check
null_pct2 = (df2.isna().sum() / len(df2) * 100).sort_values(ascending=False)
print(null_pct2[null_pct2 > 0].round(1).to_string())
print('cleaned shape:', df2.shape)

allarmi_clean = df2.copy()

flag_rischio             99.0
note_operatore           98.5
MOTIVO_ALLARME           22.8
DESCR_AEREOPORTO_PART     6.0
CITTA_PARTENZA            5.0
PAESE_PART                4.0
CODICE_PAESE_PART         3.2
TOT                       2.4
ZONA                      0.1
OCCORRENZE                0.1
cleaned shape: (5080, 19)


In [117]:
viaggiatori_clean.to_csv(DATA_DIR / 'viaggiatori_clean.csv', index=False)
allarmi_clean.to_csv(DATA_DIR / 'allarmi_clean.csv', index=False)

print('viaggiatori_clean.csv ->', viaggiatori_clean.shape)
print('allarmi_clean.csv     ->', allarmi_clean.shape)

viaggiatori_clean.csv -> (5095, 28)
allarmi_clean.csv     -> (5080, 19)


## Before / After Cleaning Summary

In [118]:
summary = pd.DataFrame({
    'dataset': ['viaggiatori', 'viaggiatori', 'allarmi', 'allarmi'],
    'state':   ['raw', 'clean', 'raw', 'clean'],
    'rows':    [len(viaggiatori), len(viaggiatori_clean), len(allarmi), len(allarmi_clean)],
    'columns': [viaggiatori.shape[1], viaggiatori_clean.shape[1], allarmi.shape[1], allarmi_clean.shape[1]],
    'total_nulls':  [viaggiatori.isna().sum().sum(), viaggiatori_clean.isna().sum().sum(),
                     allarmi.isna().sum().sum(),     allarmi_clean.isna().sum().sum()],
    'null_%': [
        round(viaggiatori.isna().mean().mean() * 100, 1),
        round(viaggiatori_clean.isna().mean().mean() * 100, 1),
        round(allarmi.isna().mean().mean() * 100, 1),
        round(allarmi_clean.isna().mean().mean() * 100, 1),
    ]
})

summary

,dataset,state,rows,columns,total_nulls,null_%
0,viaggiatori,raw,5095,33,11757,7.0
1,viaggiatori,clean,5095,28,15425,10.8
2,allarmi,raw,5080,24,11530,9.5
3,allarmi,clean,5080,19,12245,12.7


In [119]:
# null % per column before vs after, side by side
v_null_raw   = (viaggiatori.isna().mean() * 100).rename('raw')
v_null_clean = (viaggiatori_clean.isna().mean() * 100).rename('clean')
v_compare = pd.concat([v_null_raw, v_null_clean], axis=1).dropna(how='all')
v_compare = v_compare[v_compare['raw'] > 0].round(1).sort_values('raw', ascending=False)

a_null_raw   = (allarmi.isna().mean() * 100).rename('raw')
a_null_clean = (allarmi_clean.isna().mean() * 100).rename('clean')
a_compare = pd.concat([a_null_raw, a_null_clean], axis=1).dropna(how='all')
a_compare = a_compare[a_compare['raw'] > 0].round(1).sort_values('raw', ascending=False)

print('=== VIAGGIATORI: null % per column ===')
print(v_compare.to_string())
print()
print('=== ALLARMI: null % per column ===')
print(a_compare.to_string())

=== VIAGGIATORI: null % per column ===
                  raw  clean
codice_rischio   99.2   99.2
note_operatore   98.8   98.8
ESITO_CONTROLLO  25.3   25.3
NAZIONALITA       2.3    6.3
COMPAGNIA_AEREA   1.7    5.1
NUMERO_VOLO       1.4    4.0
TIPO_DOCUMENTO    1.2   35.5
GENERE            0.9   13.2

=== ALLARMI: null % per column ===
                        raw  clean
flag_rischio           99.0   99.0
note_operatore         98.5   98.5
MOTIVO_ALLARME         22.8   22.8
DESCR_AEREOPORTO_PART   2.1    6.0
CITTA_PARTENZA          2.0    5.0
PAESE_PART              1.5    4.0
CODICE_PAESE_PART       1.1    3.2


In [120]:
# drop columns with >50% nulls from both datasets
threshold = 0.50

v_drop = [c for c in viaggiatori_clean.columns if viaggiatori_clean[c].isna().mean() > threshold]
a_drop = [c for c in allarmi_clean.columns if allarmi_clean[c].isna().mean() > threshold]

print('dropping from viaggiatori:', v_drop)
print('dropping from allarmi:', a_drop)

viaggiatori_clean.drop(columns=v_drop, inplace=True)
allarmi_clean.drop(columns=a_drop, inplace=True)

dropping from viaggiatori: ['note_operatore', 'codice_rischio']
dropping from allarmi: ['note_operatore', 'flag_rischio']


In [121]:
# re-export
viaggiatori_clean.to_csv(DATA_DIR / 'viaggiatori_clean.csv', index=False)
allarmi_clean.to_csv(DATA_DIR / 'allarmi_clean.csv', index=False)

print('viaggiatori_clean:', viaggiatori_clean.shape)
print('allarmi_clean:    ', allarmi_clean.shape)

viaggiatori_clean: (5095, 26)
allarmi_clean:     (5080, 17)
